# 3. Microsoft Sentinel and Defender XDR

This notebook covers Microsoft's detection & response stack:

- **Microsoft Sentinel** — cloud-native SIEM *and* SOAR.
- **Microsoft Defender XDR** — correlated threat detection across email, endpoint, identity, and cloud apps.

We'll simulate analytics rules, KQL queries, XDR correlation, and an automated playbook response.

## 🔧 Setup (run this once)

Before running the code cells, make sure you've installed dependencies and selected the right kernel:

```bash
cd security-certs/sc-900/03-azure-security-solutions
uv sync
```

Then in **VS Code**: click the kernel picker in the top-right of this notebook and pick the `.venv` interpreter for this folder. If it doesn't show up, reload the window (`Cmd+Shift+P` → *Developer: Reload Window*).

> ℹ️ **No Azure account needed.** These notebooks *simulate* Azure services in pure Python so you can learn the concepts without any cloud cost.

---
## 1. SIEM vs SOAR

| | SIEM | SOAR |
|-|------|------|
| **Stands for** | Security Information and Event Management | Security Orchestration, Automation, and Response |
| **Purpose** | Collect, analyze, and correlate security logs | Automate incident response workflows |
| **Example action** | *"Alert: suspicious login from new country"* | *"Auto-disable user + create ticket + notify Teams"* |
| **Microsoft tool** | Microsoft Sentinel | Microsoft Sentinel (same product!) |

> 🎯 **Microsoft Sentinel is both SIEM *and* SOAR** — it collects logs *and* automates responses.

## 2. Microsoft Sentinel

Cloud-native SIEM/SOAR built on Azure. It ingests data from almost anywhere:

| Data source | Connector |
|------------|----------|
| Microsoft 365 | Built-in |
| Azure resources | Built-in (diagnostics) |
| Entra ID sign-in/audit | Built-in |
| Defender XDR incidents | Built-in |
| Firewalls (Palo Alto, Fortinet...) | Data connector |
| Linux syslog / Windows events | Azure Monitor Agent |
| Custom apps | CEF, Syslog, REST API |

### The four Sentinel capabilities

| Capability | What it does |
|-----------|-------------|
| **Collect** | Ingest data from any source at cloud scale |
| **Detect** | Find threats with built-in analytics rules and ML |
| **Investigate** | Explore incidents with interactive graphs and timelines |
| **Respond** | Automate with playbooks (Logic Apps) |

---
## 3. KQL — the query language of Sentinel

Sentinel stores data in **Log Analytics workspaces** and queries it with **KQL (Kusto Query Language)**. KQL is also used by Defender, Azure Monitor, and Azure Data Explorer.

You won't have to *write* KQL on the SC-900 exam, but reading it helps you understand what Sentinel does. The cells below mock sign-in logs and run three classic detection queries against them.

In [ ]:
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import random, json

USERS     = ['alice@contoso.com', 'bob@contoso.com', 'carol@contoso.com', 'attacker@evil.com']
LOCATIONS = ['Seattle', 'London', 'Moscow', 'Beijing', 'Office']

random.seed(42)
SIGN_IN_LOGS = []
now = datetime.now()
for _ in range(80):
    user = random.choice(USERS)
    loc  = random.choice(LOCATIONS)
    status = 'Success' if random.random() > 0.3 else 'Failure'
    hours_ago = random.randint(0, 48)
    if user == 'attacker@evil.com':
        # Make the attacker's failures cluster into a single hour so KQL #3 detects brute force.
        status, loc = 'Failure', random.choice(['Moscow', 'Beijing'])
        hours_ago = 1
    SIGN_IN_LOGS.append({
        'TimeGenerated':     (now - timedelta(hours=hours_ago)).isoformat()[:19],
        'UserPrincipalName': user,
        'Location':          loc,
        'ResultType':        status,
        'AppDisplayName':    random.choice(['Azure Portal', 'Outlook', 'Teams']),
    })

print(f'Generated {len(SIGN_IN_LOGS)} mock SigninLogs entries')
print('Sample:', json.dumps(SIGN_IN_LOGS[0], indent=2))

In [ ]:
# ----- KQL #1 — count failed sign-ins by user -----
# SigninLogs | where ResultType != "Success" | summarize count() by UserPrincipalName
print('=== KQL #1 — Failed sign-ins by user ===')
failed = Counter(l['UserPrincipalName'] for l in SIGN_IN_LOGS if l['ResultType'] == 'Failure')
for user, c in failed.most_common():
    alert = ' ⚠️ SUSPICIOUS — high failure count!' if c > 5 else ''
    print(f'  {user:<30} {c:>2} failures{alert}')

In [ ]:
# ----- KQL #2 — sign-ins from outside trusted geographies -----
# SigninLogs | where Location !in ('Seattle', 'Office', 'London')
print('=== KQL #2 — Sign-ins from unusual locations ===')
trusted = {'Seattle', 'Office', 'London'}
for l in [x for x in SIGN_IN_LOGS if x['Location'] not in trusted][:10]:
    print(f"  {l['TimeGenerated']}  {l['UserPrincipalName']:<30}  {l['Location']:<10}  {l['ResultType']}")

In [ ]:
# ----- KQL #3 — brute-force detection (>3 failures in 1 hour per user) -----
# SigninLogs
# | where ResultType == "Failure"
# | summarize FailCount=count() by UserPrincipalName, bin(TimeGenerated, 1h)
# | where FailCount > 3
print('=== KQL #3 — Brute-force detection ===')
hourly = defaultdict(int)
for l in SIGN_IN_LOGS:
    if l['ResultType'] == 'Failure':
        hourly[(l['UserPrincipalName'], l['TimeGenerated'][:13])] += 1

brute = {k: v for k, v in hourly.items() if v > 3}
if brute:
    for (user, hour), c in brute.items():
        print(f'  🚨 {user} had {c} failures at {hour}:00 — possible brute force!')
else:
    print('  No brute-force patterns in this sample.')

### Bad practice → Best practice: responding to the alert above

| ❌ Bad | 😐 Better | ✅ Best |
|-------|----------|--------|
| Nobody reads the alert until next morning | An analyst triages manually, but response takes 30 minutes | Analytics rule auto-triggers a **playbook**: disable user, open ticket, notify Teams — all in seconds |

That last column is the **SOAR** part of Sentinel. Playbooks are just Azure Logic Apps — drag-and-drop workflows with hundreds of built-in connectors.

In [ ]:
# Simulate a Sentinel playbook firing on a brute-force incident
class SentinelPlaybook:
    def __init__(self, name):
        self.name = name
        self.actions = []

    def add(self, action):
        self.actions.append(action); return self

    def run(self, incident):
        print(f'▶️  Playbook "{self.name}" triggered by incident "{incident["title"]}"')
        for i, a in enumerate(self.actions, 1):
            print(f'   {i}. {a(incident)}')


def disable_user(inc):      return f'Entra ID: disabled user {inc["user"]}'
def revoke_sessions(inc):   return f'Entra ID: revoked all active sessions for {inc["user"]}'
def notify_soc(inc):        return f'Teams: posted to #soc-alerts ({inc["severity"]} severity)'
def create_ticket(inc):     return f'ServiceNow: opened INC#{hash(inc["title"]) % 100000:05d}'
def enrich_ti(inc):         return f'Threat Intel: marked source IP {inc["src_ip"]} as known bad'

incident = {
    'title':    'Brute force: attacker@evil.com (12 failures in 1h)',
    'user':     'attacker@evil.com',
    'src_ip':   '198.51.100.77',
    'severity': 'High',
}

playbook = (
    SentinelPlaybook('Respond-BruteForce')
    .add(disable_user).add(revoke_sessions).add(enrich_ti)
    .add(notify_soc).add(create_ticket)
)
playbook.run(incident)

---
## 4. Microsoft Defender XDR

**XDR = eXtended Detection and Response.** It's a suite of Microsoft Defender products that share signals and provide unified detection across your environment.

| Defender product | What it protects |
|-----------------|-------------------|
| **Defender for Endpoint** | Devices (laptops, servers) — EDR, vulnerability management |
| **Defender for Office 365** | Email and collaboration — anti-phishing, safe attachments, safe links |
| **Defender for Identity** | On-prem AD — detects lateral movement, pass-the-hash, recon |
| **Defender for Cloud Apps** | SaaS apps — shadow IT, app governance, session controls (CASB) |
| **Defender Vulnerability Management** | Discover + remediate vulnerabilities |
| **Defender Threat Intelligence** | Threat articles, IOCs, attacker profiles |

All of these are managed from the unified portal at **security.microsoft.com**.

### Bad practice → Best practice: alert fatigue

| ❌ Bad (no XDR) | ✅ Best (XDR) |
|-----------------|---------------|
| Each product sends its own alerts. Analyst drowns in 50 alerts per attack. | Defender correlates related alerts into a **single incident** with a kill-chain timeline. |
| Manual triage per product console | One portal, one incident, one response |

In [ ]:
# Simulate XDR correlating alerts from 5 products into a single incident.
ALERTS = [
    {'source': 'Defender for Office 365', 'title': 'Phishing email delivered to alice@contoso.com',        'time': '09:00'},
    {'source': 'Defender for Endpoint',   'title': 'Suspicious PowerShell on ALICE-LAPTOP',                 'time': '09:05'},
    {'source': 'Entra ID Protection',     'title': 'Atypical token usage for alice@contoso.com',           'time': '09:07'},
    {'source': 'Defender for Cloud Apps', 'title': 'Mass file download from SharePoint by alice',          'time': '09:10'},
    {'source': 'Defender for Identity',   'title': 'Lateral movement attempt from ALICE-LAPTOP',           'time': '09:15'},
]

print('=== ❌ Without XDR: 5 separate alerts across 5 consoles ===')
for a in ALERTS:
    print(f'  [{a["time"]}] {a["source"]:<28} {a["title"]}')

print('\n=== ✅ With XDR: 1 correlated incident ===')
kill_chain = [
    ('📧', 'Initial access',   'phishing email (Office 365)'),
    ('💻', 'Execution',        'malicious PowerShell (Endpoint)'),
    ('🔑', 'Credential access','token abuse (Entra ID)'),
    ('📁', 'Exfiltration',     'mass download (Cloud Apps)'),
    ('↔️', 'Lateral movement','spread to other devices (Identity)'),
]
print('📋 Incident #4821  severity=HIGH  status=Active')
print('   Title: Multi-stage attack on alice@contoso.com')
print('   Kill chain:')
for i, (icon, stage, detail) in enumerate(kill_chain, 1):
    print(f'     {i}. {icon}  {stage:<18} — {detail}')
print('   Entities: alice@contoso.com, ALICE-LAPTOP, 3 SharePoint sites')
print('   🤖 Automatic attack disruption: user disabled, device isolated')

### Real-world attack scenarios XDR is known for

1. **Business email compromise (BEC)** — Office 365 flags the phishing email → Entra ID notices new inbox rule forwarding finance mail → incident created automatically.
2. **Ransomware** — Endpoint sees mass file rename → Cloud Apps sees suspicious download → containment fires.
3. **Insider threat** — Cloud Apps sees unusual sharing behavior → Identity sees after-hours access to sensitive OUs → unified investigation.

---
## 5. Sentinel vs Defender XDR — how they compare

| | Microsoft Sentinel | Defender XDR |
|-|-------------------|---------------|
| **Type** | SIEM / SOAR | XDR |
| **Scope** | Any log source (Microsoft + 3rd-party + custom) | Microsoft 365 + endpoints + identity + cloud apps |
| **Data** | Logs, events, threat intel | Alerts from Defender products |
| **Best for** | SOC teams needing unified visibility across *everything* | Automated detection inside the Microsoft ecosystem |
| **Portal** | Azure portal (Sentinel workspace) | security.microsoft.com |

They **work together**: Defender XDR incidents can flow into Sentinel so you can correlate them with non-Microsoft data (e.g., your firewall logs).

### Exam tip

- Sentinel = SIEM + SOAR, cloud-native, any data source.
- Defender XDR = correlated threat detection across Microsoft products.
- Sentinel can **ingest** Defender XDR incidents.
- The unified Defender portal is at **security.microsoft.com**.

---
## Summary

| Concept | Key fact |
|---------|----------|
| **SIEM** | Collect and analyze security logs |
| **SOAR** | Automate incident response |
| **Sentinel** | Cloud-native SIEM + SOAR. Uses KQL. Playbooks = Logic Apps. |
| **KQL** | Query language of Sentinel / Defender / Azure Monitor |
| **Defender XDR** | Correlated detection across endpoint, email, identity, cloud apps |
| **Defender for Endpoint** | EDR for devices |
| **Defender for Office 365** | Anti-phishing for email |
| **Defender for Identity** | On-prem AD protection |
| **Defender for Cloud Apps** | SaaS protection (shadow IT, CASB) |

**Next lab**: [04 — Compliance and Purview](../../04-compliance-and-purview/)